# Build AI: train the 16×16 pixel-art model

Runs on a free **Kaggle** (Settings → Accelerator → GPU T4, Internet on) or **Google Colab** (Runtime → GPU) machine.
Run the cells top to bottom. At the end you download **`pixel16.bin`** (about 6 MB) and import it in the Build AI app, or send it to Claude to bundle into the APK.

Training saves progress. If the session stops, run everything again and it continues where it left off (on Kaggle, keep `/kaggle/working`).

In [ ]:
import os
WORK = '/kaggle/working' if os.path.exists('/kaggle') else '/content'
os.chdir(WORK)
!git clone -q -b ccr-8b2e2ac2-hebcme https://github.com/mirzahosseinimoqaddam-cmd/PriorCenter.git || (cd PriorCenter && git pull -q)
!pip -q install datasets scikit-learn
os.chdir(f'{WORK}/PriorCenter/build-ai/trainer')
DATASET = 'Angais/pixel-art-atlas-1.4m'
DATA, RUN = f'{WORK}/data16', f'{WORK}/run16'

## 1. Look at the dataset first
Shows the columns, a few examples and the sizes of the first 2,000 images, so we know how many are true 16×16 art and whether they have captions.
If the dataset needs a login, add a Hugging Face token as the secret `HF_TOKEN` (Kaggle: Add-ons → Secrets; Colab: the key icon).

In [ ]:
from datasets import load_dataset
from collections import Counter
from prepare import to_grid
from PIL import Image
import io
ds = load_dataset(DATASET, split='train', streaming=True)
print(ds.features)
sizes, grids, n = Counter(), Counter(), 0
for row in ds:
    if n < 3: print({k: (v if not isinstance(v, (Image.Image, dict, bytes)) else type(v).__name__) for k, v in row.items()})
    img = next((v for v in row.values() if isinstance(v, Image.Image)), None)
    if img is None:
        v = next(v for v in row.values() if isinstance(v, dict) and 'bytes' in v); img = Image.open(io.BytesIO(v['bytes']))
    sizes[img.size] += 1
    for s in (16, 32, 64):
        if to_grid(img, s) is not None: grids[s] += 1; break
    n += 1
    if n >= 2000: break
print('most common image sizes:', sizes.most_common(10))
print('of 2000 images, fits after undoing upscaling into: ', dict(grids))

## 2. Prepare the 16×16 training data (about 20–40 minutes)
All sources are free to use (licences in build-ai/README.md). A source that fails to load is skipped, the rest continue.
Images with non-halal captions (list in `prepare.py`, `NOT_HALAL`) or with no caption are left out at training time.

In [ ]:
HF_SOURCES = [
    DATASET,                          # 1.4M pixel-art images
    'ptx0/free-to-use-pixelart',      # MIT, PixilArt free-to-use
    'jainr3/diffusiondb-pixelart',    # CC0, pixel-art samples from an AI generator
]
# (repo, folder inside it, sparse path or None); licences in build-ai/README.md
GIT = [
    ('tstamborski/pixelart-icons', 'png', None),                                  # CC0
    ('markjames/famfamfam-silk-icons', 'icons', None),                            # CC-BY 2.5
    ('jdecked/twemoji', 'assets/72x72', '/assets/72x72/*'),                        # CC-BY 4.0
    ('iwenzhou/kenney', 'Art (5190 files)', None),                                # CC0
    ('PixelRubro/Pixel-Art-Prototyping-CC0-Collection', '', None),                # CC0
    ('doficia/project-cordon-sprites', 'sprites', None),                          # CC0
    ('MarcelloCalligaris/Pixel_Icons', '', None),                                 # CC0
    ('LoveBootCaptain/unicornhat_weather_icons', '', None),                       # MIT
]
src_dir = f'{WORK}/src'; os.makedirs(src_dir, exist_ok=True)
folders = []
for repo, sub, sparse in GIT:
    d = f'{src_dir}/' + repo.split('/')[1]
    if not os.path.exists(d):
        if sparse:
            os.system(f'git clone -q --depth 1 --filter=blob:none --sparse https://github.com/{repo}.git {d} && cd {d} && git sparse-checkout set --no-cone "{sparse}"')
        else:
            os.system(f'git clone -q --depth 1 https://github.com/{repo}.git {d}')
    folders.append(os.path.join(d, sub))
# 16x16 sprites from the open DeepLearning.AI diffusion course (stored with Git LFS)
sp = f'{src_dir}/sprites16'
if not os.path.exists(sp):
    base = 'https://media.githubusercontent.com/media/Shamik-07/CodeSnippets/main/deeplearning.ai_courses/evaluating_and_debugging_gen_ai_wnb/data'
    !wget -q {base}/sprites_1788_16x16.npy {base}/sprite_labels_nc_1788_16x16.npy -P {src_dir}
    !python import_npy_sprites.py {src_dir}/sprites_1788_16x16.npy {src_dir}/sprite_labels_nc_1788_16x16.npy {sp}
folders.append(sp)
args = ' '.join(f'--hf {s}' for s in HF_SOURCES) + ' ' + ' '.join(f'--folder "{f}"' for f in folders)
if not os.path.exists(f'{DATA}/tokens.npy'):
    !python prepare.py --out {DATA} --size 16 {args}
print(open(f'{DATA}/stats.json').read())

## 3. Captions
The model learns words from captions. If many images have no caption, this writes them with an open image-captioning model (BLIP). It takes about 1 hour per 100k images on a T4.

In [ ]:
import json
caps = json.load(open(f'{DATA}/captions.json'))
empty = sum(1 for c in caps if not c)
print(f'{empty} of {len(caps)} images have no caption')
for c in caps[:15]: print(' ', c)
if empty > 0.2 * len(caps):
    !pip -q install transformers
    !python caption.py --data {DATA}

## 4. Train (about 2–4 hours on a T4)
Example pictures are saved as `samples_*.png` in the run folder every 2,000 steps.

In [ ]:
!python train.py --data {DATA} --out {RUN} --steps 60000 --batch 256 --dim 256 --layers 8 --heads 8

In [ ]:
from IPython.display import Image as Show, display
import glob
last = sorted(glob.glob(f'{RUN}/samples_*.png'), key=lambda p: int(p.split('_')[-1][:-4]))[-1]
print(last); display(Show(last))

## 5. Export for the phone and check it

In [ ]:
!python export.py --ckpt {RUN}/ckpt.pt --out {WORK}/pixel16.bin --name "Build AI pixel 16"
!python test_parity.py {WORK}/pixel16.bin

In [ ]:
# download pixel16.bin
if os.path.exists('/kaggle'):
    from IPython.display import FileLink; display(FileLink('pixel16.bin'))
else:
    from google.colab import files; files.download(f'{WORK}/pixel16.bin')